# 01 - API Extraction

Ce notebook documente la phase d'extraction depuis **TMDb** et l'enrichissement via **OMDb**.

Objectifs :
- vérifier les variables d'environnement ;
- lancer une extraction réelle si besoin ;
- charger la dernière extraction disponible ;
- faire un premier contrôle de structure.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == 'notebooks':
    ROOT_DIR = ROOT_DIR.parent

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

RAW_DIR = ROOT_DIR / 'data' / 'raw'
LATEST_CSV = RAW_DIR / 'tmdb_animation_movies_latest.csv'
LATEST_JSONL = RAW_DIR / 'tmdb_animation_movies_latest.jsonl'

print('ROOT_DIR =', ROOT_DIR)
print('TMDB configured =', bool(os.getenv('TMDB_BEARER_TOKEN') or os.getenv('TMDB_API_KEY')))
print('OMDB configured =', bool(os.getenv('OMDB_API_KEY')))
print('Latest CSV exists =', LATEST_CSV.exists())
print('Latest JSONL exists =', LATEST_JSONL.exists())

## Lancer une extraction réelle

Décommente la cellule suivante uniquement si tu veux relancer l'extraction depuis le notebook.

In [ ]:
# from src.api.extract_movies import extract_movies
# result = extract_movies(
#     start_year=2015,
#     end_year=2025,
#     max_pages_per_year=5,
#     original_language=None,
#     include_omdb=True,
# )
# result

In [ ]:
df_raw = pd.read_csv(LATEST_CSV)
print(df_raw.shape)
df_raw.head(3)

In [ ]:
df_raw.info()

In [ ]:
summary = pd.DataFrame({
    'dtype': df_raw.dtypes.astype(str),
    'missing_count': df_raw.isna().sum(),
    'missing_ratio': df_raw.isna().mean().round(3),
    'n_unique': df_raw.nunique(dropna=True),
}).sort_values(by='missing_ratio', ascending=False)
summary.head(20)

In [ ]:
quality_checks = {
    'rows': len(df_raw),
    'duplicate_tmdb_id': int(df_raw['tmdb_id'].duplicated().sum()) if 'tmdb_id' in df_raw.columns else None,
    'duplicate_title_release': int(df_raw.duplicated(subset=['title', 'release_date']).sum()),
    'share_budget_zero': round((df_raw['budget'] == 0).mean(), 3),
    'share_revenue_zero': round((df_raw['revenue'] == 0).mean(), 3),
}
quality_checks